# AC1 — 자동 교정: 판정기가 경고한 문장을 LLM이 고쳐 쓰면 환각이 줄어드는가

## 왜 하나

지금 게이트웨이는 환각을 **경고만** 하고 응답을 바꾸지 않는다. 경고가 서비스 가치가 되려면 "경고 → 고치기"까지 이어져야 한다.
두 가지를 확인한다. ① 판정기가 짚은 문장을 LLM이 근거와 대조해 고쳐 쓰면 **환각 응답이 실제로 줄어드는가**.
② 그 효과가 "모든 응답을 다시 검토시키기"와 비교해 **판정기 덕분에 싸게** 얻어지는가(판정기의 가치).

## 무엇을 하나

- **응답**: RAGTruth **dev-L 원본 응답**(train 분할, 판정기 학습·보정에 쓰이지 않은 근거) 과제별 150개 = 450개, `Random(2040)`
- **판정**: 배포 판정기 **v1.0**(F1 base + I2 조합, 8토큰 + 문장 끝, **임계값 B 2.7385**). 경보 지점이 있는 문장 = 경고 문장
- **세 팔**
  - `orig`: 원본 그대로
  - `tgt`: **경고 응답만** `claude-haiku-5-5`가 고쳐 씀 — 경고 문장 앞에 `[CHECK]` 표시, 그 문장만 근거와 대조해 고치거나 뺌
  - `blind`: **모든 응답**을 같은 모델이 표시 없이 "근거와 대조해 근거 없는 내용을 고쳐라"로 고쳐 씀 (대조군)
- **평가**(`claude-opus-5-5`, 어느 팔인지 가림, 같은 글은 한 번만): 충실성(R4 문장 S/H/A → H 하나라도 있으면 환각 응답), 정답성(QA만, T1b yes/partial/no), 정보 손실(고친 글만, "근거가 뒷받침하는 원본 정보를 빼거나 약화했나")

## 사전 정의 기준 (SSOT §6 AC1, 11/26 고정)

- **AC1-0 유효성**: 세 팔 모두 고쳐 쓰기·평가 성공 응답 ≥ 405, 경고 응답 ≥ 60. 미달이면 판정 보류
- **AC1-1 (주, 효과)**: 환각 응답 비율 상대 변화 (tgt − orig)/orig ≤ **−15%**, 그리고 차이의 95% 상한 < 0
- **AC1-2 (부작용)**: QA `yes` tgt − orig ≥ −0.05(하한 ≥ −0.10), 그리고 tgt에서 고친 응답 중 정보 손실 ≤ 15%
- **AC1-3 (판정기의 가치)**: 고친 응답 비율 ≤ 30%, 그리고 감소폭 (orig − tgt) ≥ 0.5 × (orig − blind)
- **채택 = AC1-0~3** → 게이트웨이 "응답 끝 교정" 기능을 SPEC부터 설계(선택 기능). AC1-1·2만 충족이면 "교정 효과는 있으나 판정기 안내의 이점은 확인 안 됨"
- 응답 단위 쌍체 부트스트랩 2,000회(시드 2040). **API 상한 $25**(제출 전 예상 > $22면 멈춤)

## 실행 방법

1. 왼쪽 열쇠 아이콘(보안 비밀)에 `ANTHROPIC_API_KEY`, 노트북 접근 켜기
2. 런타임 유형 **GPU**(A100·L4·T4 아무거나 — 판정기 채점용, 속도는 재지 않음)
3. **모두 실행**. 배치는 보통 수십 분~1시간. 끊기면 다시 모두 실행 — 드라이브 `ac1/`에서 이어서 한다
4. 끝나면 `ac1_results.zip`(원문 없음)이 내려받아진다


## 1. 환경 준비

In [ ]:
SRC = {'judge': '"""문장 판정기.\n\n입력 방식은 판정기 폴더의 `judge_config.json`이 정한다 (SPEC §4).\n  retrieval (F1 base): 근거 → 40단어 조각 → 임베딩 (요청당 1회, prepare)\n                       판정 텍스트 → 임베딩 → 유사한 조각 top-k (원문 순서) → 교차 인코더 [근거 조각, 문장 앞부분]\n  full      (I2식)    : 교차 인코더 [근거 전체, 응답 처음부터 판정 지점까지]. 근거 검색 없음\n점수는 logit(클수록 환각). 임계값은 재판정 간격(stride)별로 설정 파일의 값을 쓴다.\n설정 파일이 "type": "combo"이면 하위 판정기 여러 개의 표준화 점수를 평균하는 ComboJudge를 만든다 (배포 판정기 I3).\n"""\nimport contextlib\nimport json\nimport os\nimport threading\nimport time\nfrom concurrent.futures import ThreadPoolExecutor\nfrom dataclasses import dataclass\n\nimport torch\n\n\n@dataclass\nclass Verdict:\n    score: float          # logit, 클수록 환각\n    hallucinated: bool\n    judge_ms: float       # 검색 + 판정 시간\n\n\n@dataclass\nclass RefIndex:\n    chunks: list\n    emb: object           # torch.Tensor [n_chunks, d] (retrieval) / None (full)\n    prepare_ms: float\n    text: str = \'\'        # 근거 원문 (full 방식)\n\n\ndef chunk_text(text, n):\n    w = text.split()\n    return [\' \'.join(w[i:i + n]) for i in range(0, max(len(w), 1), n)] or [text]\n\n\ndef _stride_key(stride):\n    return \'end\' if not stride else str(stride)\n\n\ndef read_config(model_dir):\n    p = os.path.join(model_dir, \'judge_config.json\')\n    return json.load(open(p)) if os.path.isfile(p) else {}\n\n\ndef input_mode(cfg):\n    """설정 파일에서 입력 방식을 정한다. 명시(`input_mode`)가 없으면 I2 노트북 형식(retrieval: null)을 full로 본다."""\n    if cfg.get(\'input_mode\'):\n        return cfg[\'input_mode\']\n    inp = cfg.get(\'input\') or {}\n    return \'full\' if isinstance(inp, dict) and \'retrieval\' in inp and inp[\'retrieval\'] is None else \'retrieval\'\n\n\nclass StudentJudge:\n    def __init__(self, model_dir, embedder=\'sentence-transformers/all-MiniLM-L6-v2\', k=8, max_len=512,\n                 chunk_words=40, threshold=0.0, device=\'\', mode=\'retrieval\', thresholds=None, threshold_key=\'svc10_orig\',\n                 default_stride=0):\n        from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer\n        self.device = device or (\'cuda\' if torch.cuda.is_available() else \'cpu\')\n        self.mode = mode\n        self.tok = AutoTokenizer.from_pretrained(model_dir)\n        cfg = AutoConfig.from_pretrained(model_dir)\n        kw = {}\n        if getattr(cfg, \'model_type\', \'\') == \'modernbert\':\n            # 새 transformers는 reference_compile을 모델 인자로 받지 않으므로 설정에 넣는다\n            if hasattr(cfg, \'reference_compile\'):\n                cfg.reference_compile = False\n            kw[\'attn_implementation\'] = \'sdpa\'\n        # 가중치가 fp16으로 저장돼 있어도 fp32로 올린다 (CPU 판정, 노트북 점수와 일치). GPU에서는 autocast가 반정밀도로 돌린다.\n        self.model = AutoModelForSequenceClassification.from_pretrained(model_dir, config=cfg, **kw).float().to(self.device).eval()\n        if mode == \'retrieval\' and isinstance(embedder, str):\n            from sentence_transformers import SentenceTransformer\n            embedder = SentenceTransformer(embedder, device=self.device)\n        self.embedder = embedder\n        self.k, self.max_len, self.chunk_words, self.threshold = k, max_len, chunk_words, threshold\n        self.thresholds, self.threshold_key = thresholds or {}, threshold_key\n        self.default_stride = default_stride\n        self._lock = threading.Lock()     # 모델 1개를 여러 요청이 공유하므로 직렬화\n\n    def threshold_for(self, stride, key=None):\n        """재판정 간격별 임계값 (설정 파일 thresholds[간격][key]). 없으면 기본 임계값."""\n        return (self.thresholds.get(_stride_key(stride)) or {}).get(key or self.threshold_key, self.threshold)\n\n    def _embed(self, texts):\n        return self.embedder.encode(texts, convert_to_tensor=True, normalize_embeddings=True,\n                                    show_progress_bar=False)\n\n    def prepare(self, reference):\n        t0 = time.perf_counter()\n        if self.mode == \'full\':\n            return RefIndex([], None, (time.perf_counter() - t0) * 1000, reference)\n        chunks = chunk_text(reference, self.chunk_words)\n        with self._lock:\n            emb = self._embed(chunks)\n        return RefIndex(chunks, emb, (time.perf_counter() - t0) * 1000, reference)\n\n    def _pair(self, ref, text, prior):\n        if self.mode == \'full\':\n            return ref.text, (prior or \'\') + text\n        e = self._embed([text])[0]\n        top = torch.topk(ref.emb @ e, min(self.k, ref.emb.shape[0])).indices.tolist()\n        return \' ... \'.join(ref.chunks[i] for i in sorted(top)), text\n\n    @torch.no_grad()\n    def score(self, ref, text, prior=\'\', stream=None):\n        """logit 하나. stream(CUDA 스트림)을 주면 그 스트림에서 계산하고 그 스트림만 기다린다(조합 판정기의 동시 실행용)."""\n        ctx = torch.cuda.stream(stream) if (stream is not None and self.device == \'cuda\') else contextlib.nullcontext()\n        with self._lock, ctx:\n            a, b = self._pair(ref, text, prior)\n            enc = self.tok([a], [b], truncation=\'only_first\', max_length=self.max_len,\n                           return_tensors=\'pt\').to(self.device)\n            with torch.autocast(self.device, enabled=(self.device == \'cuda\')):\n                score = float(self.model(**enc).logits.squeeze().float().cpu())   # .cpu()가 현재 스트림을 기다림\n            if self.device == \'cuda\' and stream is None:\n                torch.cuda.synchronize()\n        return score\n\n    def judge(self, ref, text, prior=\'\', threshold=None):\n        """text: 판정할 문장(또는 문장 앞부분, 앞뒤 공백 제거), prior: 응답에서 그 문장 앞까지의 원문."""\n        t0 = time.perf_counter()\n        score = self.score(ref, text, prior)\n        thr = self.threshold if threshold is None else threshold\n        return Verdict(score, score >= thr, (time.perf_counter() - t0) * 1000)\n\n\n@dataclass\nclass ComboRef:\n    refs: list            # 하위 판정기마다 RefIndex (members 순서)\n    prepare_ms: float\n\n\nclass ComboJudge:\n    """하위 판정기 점수를 (score − mean) / std로 표준화해 평균한다 (SPEC §4, 노트북 I3·L3의 combine과 같은 식).\n    판정 1회에서 하위 판정기를 동시에 돌린다(스레드, GPU면 판정기마다 별도 CUDA 스트림)."""\n\n    mode = \'combo\'\n\n    def __init__(self, members, thresholds=None, threshold=0.0, threshold_key=\'svcB\', default_stride=0):\n        # members: [(이름, 판정기, mean, std), ...]\n        assert members and all(sd > 0 for _, _, _, sd in members), \'하위 판정기와 양수 표준편차가 필요\'\n        self.members = members\n        self.thresholds, self.threshold, self.threshold_key = thresholds or {}, threshold, threshold_key\n        self.default_stride = default_stride\n        self._pool = ThreadPoolExecutor(len(members), thread_name_prefix=\'combo\')\n        self._streams = [torch.cuda.Stream() if getattr(j, \'device\', \'\') == \'cuda\' else None for _, j, _, _ in members]\n\n    @property\n    def embedder(self):\n        # 컬렉션 검색 임베더를 비웠을 때 같이 쓸 임베더: retrieval 방식 하위 판정기의 것\n        return next((e for _, j, _, _ in self.members for e in [getattr(j, \'embedder\', None)] if hasattr(e, \'encode\')), None)\n\n    def threshold_for(self, stride, key=None):\n        return (self.thresholds.get(_stride_key(stride)) or {}).get(key or self.threshold_key, self.threshold)\n\n    def prepare(self, reference):\n        t0 = time.perf_counter()\n        refs = [j.prepare(reference) for _, j, _, _ in self.members]\n        return ComboRef(refs, (time.perf_counter() - t0) * 1000)\n\n    @staticmethod\n    def _member_score(j, ref, text, prior, stream):\n        if hasattr(j, \'score\'):\n            return j.score(ref, text, prior, stream=stream)\n        return j.judge(ref, text, prior).score                  # 대체 판정기(테스트)\n\n    def judge(self, ref, text, prior=\'\', threshold=None):\n        t0 = time.perf_counter()\n        futs = [self._pool.submit(self._member_score, j, r, text, prior, st)\n                for (_, j, _, _), r, st in zip(self.members, ref.refs, self._streams)]\n        raw = [f.result() for f in futs]                         # 하위 판정기 예외는 그대로 올라가 fail-open으로 처리됨\n        score = sum((x - m) / sd for x, (_, _, m, sd) in zip(raw, self.members)) / len(self.members)\n        thr = self.threshold if threshold is None else threshold\n        return Verdict(score, score >= thr, (time.perf_counter() - t0) * 1000)\n\n\ndef _load_combo(settings, cfg):\n    base = settings.judge_model_dir\n    std, members = cfg[\'standardize\'], []\n    for name, sub in cfg[\'members\'].items():\n        d = sub if os.path.isabs(sub) else os.path.join(base, sub)\n        if not os.path.isfile(os.path.join(d, \'config.json\')):\n            raise FileNotFoundError(f\'조합 하위 판정기 {name} 폴더가 없음: {d}\')\n        mcfg = read_config(d)\n        j = StudentJudge(d, embedder=mcfg.get(\'embedder\', \'sentence-transformers/all-MiniLM-L6-v2\'), k=mcfg.get(\'k\', 8),\n                         max_len=mcfg.get(\'max_len\', 512), chunk_words=mcfg.get(\'chunk_words\', 40),\n                         device=settings.judge_device, mode=input_mode(mcfg))\n        members.append((name, j, float(std[name][\'mean\']), float(std[name][\'std\'])))\n    dflt = cfg.get(\'default\') or {}\n    kind = dflt.get(\'kind\', \'end\')\n    fixed = settings.judge_threshold\n    return ComboJudge(members, thresholds={} if fixed is not None else cfg.get(\'thresholds\', {}),\n                      threshold=fixed if fixed is not None else cfg.get(\'threshold\', 0.0),\n                      threshold_key=dflt.get(\'threshold\', \'svcB\'), default_stride=0 if kind == \'end\' else int(kind))\n\n\ndef load_judge(settings):\n    """판정기 폴더의 judge_config.json을 읽어 StudentJudge(또는 "type": "combo"면 ComboJudge)를 만든다.\n    환경 변수로 지정한 값이 설정 파일보다 우선한다."""\n    cfg = read_config(settings.judge_model_dir)\n    if cfg.get(\'type\') == \'combo\':\n        return _load_combo(settings, cfg)\n    pick = lambda env, key, default: env if env is not None else cfg.get(key, default)\n    dflt = (cfg.get(\'default\') or {})\n    kind = dflt.get(\'kind\', \'end\')\n    return StudentJudge(settings.judge_model_dir,\n                        embedder=pick(settings.judge_embedder, \'embedder\', \'sentence-transformers/all-MiniLM-L6-v2\'),\n                        k=pick(settings.judge_k, \'k\', 8), max_len=pick(settings.judge_max_len, \'max_len\', 512),\n                        chunk_words=pick(settings.judge_chunk_words, \'chunk_words\', 40),\n                        threshold=pick(settings.judge_threshold, \'threshold\', 0.0), device=settings.judge_device,\n                        mode=input_mode(cfg),\n                        # 임계값을 환경 변수로 고정하면 간격별 임계값은 쓰지 않는다\n                        thresholds={} if settings.judge_threshold is not None else cfg.get(\'thresholds\', {}),\n                        threshold_key=dflt.get(\'threshold\', \'svc10_orig\'),\n                        default_stride=0 if kind == \'end\' else int(kind))\n\n\nclass LexicalJudge:\n    """모델 없이 파이프라인을 시험하기 위한 대체 판정기 (테스트·데모 전용).\n    근거에 없는 단어 비율을 점수로 쓴다. 판별력은 없다고 봐야 한다."""\n\n    default_stride = 0\n\n    def __init__(self, threshold=0.5, delay_ms=0.0, fail=None):\n        self.threshold, self.delay_ms = threshold, delay_ms\n        self.fail = fail          # 테스트용: 이 함수가 True를 돌려주는 판정에서 예외를 낸다\n\n    def threshold_for(self, stride, key=None):\n        return self.threshold\n\n    def prepare(self, reference):\n        return RefIndex([reference], {w.lower().strip(\'.,\') for w in reference.split()}, 0.0, reference)\n\n    def judge(self, ref, text, prior=\'\', threshold=None):\n        t0 = time.perf_counter()\n        if self.fail is not None and self.fail(text):\n            raise RuntimeError(\'judge failure (test)\')\n        if self.delay_ms:\n            time.sleep(self.delay_ms / 1000)\n        words = [w.lower().strip(\'.,\') for w in text.split() if w.strip(\'.,\')]\n        score = sum(w not in ref.emb for w in words) / max(len(words), 1)\n        thr = self.threshold if threshold is None else threshold\n        return Verdict(score, score >= thr, (time.perf_counter() - t0) * 1000)\n'}
import os, sys, json, time, random, subprocess, shutil, pickle
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'anthropic', 'sentence-transformers', 'tiktoken', 'pysbd', '-U', 'transformers>=4.48'], check=True)
subprocess.run('git clone --depth 1 https://github.com/ParticleMedia/RAGTruth.git 2>/dev/null || true', shell=True)
import numpy as np, pandas as pd, torch
from google.colab import drive
drive.mount('/content/drive')
os.makedirs('/content/hg/hallugate', exist_ok=True)
open('/content/hg/hallugate/__init__.py', 'w').write('')
for n, s in SRC.items(): open(f'/content/hg/hallugate/{n}.py', 'w').write(s)
sys.path.insert(0, '/content/hg')
from hallugate.judge import StudentJudge, read_config, input_mode

OUT = '/content/drive/MyDrive/Colab Notebooks/HalluGate'
F_DIR, I_DIR = os.path.join(OUT, 'final', 'judge_base'), os.path.join(OUT, 'i2', 'judge_i2')
R0_DIR, AC1_DIR = os.path.join(OUT, 'r0'), os.path.join(OUT, 'ac1')
os.makedirs(AC1_DIR, exist_ok=True)
for d in [F_DIR, I_DIR]:
    assert os.path.exists(os.path.join(d, 'judge_config.json')), f'{d}/judge_config.json 없음'
assert os.path.exists(os.path.join(R0_DIR, 'r0_items.pkl')), 'r0/r0_items.pkl 없음 (dev-L 분할)'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
GPU = torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'cpu'
COMBO = {'F_mean': -3.6794357299804688, 'F_std': 2.1870224475860596, 'I_mean': -4.456174373626709, 'I_std': 2.400838851928711, 'threshold': 2.7385038673533924}      # 판정기 v1.0 (configs/judge_combo_v1.json)
THR = COMBO['threshold']
def save_json(o, p): json.dump(o, open(p + '.tmp', 'w'), ensure_ascii=False); os.replace(p + '.tmp', p)
def load_json(p): return json.load(open(p)) if os.path.exists(p) else None
print('device', GPU, '| 임계값 B', THR)

## 2. dev-L 원본 응답 450개 (과제별 150)

In [ ]:
# 판정 입력: RAGTruth train 응답의 문장 끝 + 8토큰 지점 (게이트웨이와 같은 cl100k 토큰, 문장 시작부터)
import pysbd, tiktoken
ENC = tiktoken.get_encoding('cl100k_base')
SEG = pysbd.Segmenter(language='en', clean=False, char_span=True)

def build_reference(src):
    si, tt = src['source_info'], src['task_type']
    if tt == 'QA':
        return f"Question: {si.get('question','')}\n{si.get('passages','')}"
    if tt == 'Summary':
        return si if isinstance(si, str) else json.dumps(si, ensure_ascii=False)
    return '\n'.join(f"{k}: {json.dumps(v, ensure_ascii=False) if isinstance(v,(dict,list)) else v}"
                     for k, v in (si.items() if isinstance(si, dict) else []))

def points_of(text, s=8):
    # (문장 시작 글자, 지점 글자 끝) 목록: 문장마다 s토큰째 + 문장 끝
    ids = ENC.encode(text, disallowed_special=())
    _, starts = ENC.decode_with_offsets(ids)
    ends = starts[1:] + [len(text)]
    out = []
    for sp in SEG.segment(text):
        if not sp.sent.strip(): continue
        a = sp.start + (len(sp.sent) - len(sp.sent.lstrip()))
        tok = [e for st, e in zip(starts, ends) if a <= st < sp.end or (st < a < e)]
        for k, e in enumerate(tok, 1):
            if k % s == 0 or k == len(tok):
                out.append((a, min(e, sp.end)))
    return out

def make_inputs(responses, sources, n_resp=300, n_pts=300, seed=2027):
    rs = random.Random(seed)
    pool = [r for r in responses if r['split'] == 'train']
    cands = []
    for r in rs.sample(pool, min(n_resp, len(pool))):
        ref = build_reference(sources[r['source_id']])
        for a, cut in points_of(r['response']):
            text = r['response'][a:cut].strip()
            if text: cands.append({'ref': ref, 'text': text, 'prior': r['response'][:a], 'task': sources[r['source_id']]['task_type']})
    return rs.sample(cands, min(n_pts, len(cands)))

SEED_AC, PER_TASK = 2040, 150                       # 사전 고정 (SSOT §6 AC1)
TASKS = ['QA', 'Summary', 'Data2txt']

def sample_items(r0_dev, responses, sources):
    # dev-L(R0) 원본 응답에서 과제별 150개. 전부 train 분할인지 확인
    by_id = {str(r['id']): r for r in responses}
    rs = random.Random(SEED_AC)
    items = []
    for t in TASKS:
        ids = sorted({str(x['id']) for x in r0_dev if x['task_type'] == t})
        for rid in rs.sample(ids, min(PER_TASK, len(ids))):
            r = by_id[rid]; src = sources[r['source_id']]
            assert r['split'] == 'train', f'test 분할 응답이 섞임: {rid}'
            si = src['source_info'] if isinstance(src['source_info'], dict) else {}
            items.append({'rid': rid, 'sid': r['source_id'], 'task': t, 'gen_model': r.get('model'), 'text': r['response'],
                          'ref': build_reference(src),
                          'q': si.get('question', '') if t == 'QA' else None, 'passages': si.get('passages', '') if t == 'QA' else None})
    assert len({x['rid'] for x in items}) == len(items)
    return items

BASE = 'RAGTruth/dataset'
responses = [json.loads(l) for l in open(f'{BASE}/response.jsonl')]
sources = {json.loads(l)['source_id']: json.loads(l) for l in open(f'{BASE}/source_info.jsonl')}
R0P = pickle.load(open(os.path.join(R0_DIR, 'r0_items.pkl'), 'rb'))
ITEMS = sample_items(R0P['items']['devL'], responses, sources)
print(f'응답 {len(ITEMS)}개 |', pd.Series([x['task'] for x in ITEMS]).value_counts().to_dict(),
      '| 생성 모델', pd.Series([x['gen_model'] for x in ITEMS]).value_counts().to_dict())

## 3. 판정기 v1.0으로 경고 문장 찾기 (GPU)

In [ ]:
import contextlib, threading
from concurrent.futures import ThreadPoolExecutor

def make_judge(d, device):
    # load_judge와 같은 값으로 (판정기 폴더의 judge_config.json)
    cfg = read_config(d)
    return StudentJudge(d, embedder=cfg.get('embedder', 'sentence-transformers/all-MiniLM-L6-v2'), k=cfg.get('k', 8),
                        max_len=cfg.get('max_len', 512), chunk_words=cfg.get('chunk_words', 40), device=device,
                        mode=input_mode(cfg))

def combine(sf, si):
    # I3 조합: 두 점수를 cal-L 멀쩡한 응답 기준으로 표준화해 평균
    return ((sf - COMBO['F_mean']) / COMBO['F_std'] + (si - COMBO['I_mean']) / COMBO['I_std']) / 2

def judge_seq(jF, jI, refF, refI, x):
    # 차례로: 게이트웨이 judge() 두 번 (각각 토크나이즈·검색·순전파·동기화 포함)
    return jF.judge(refF, x['text'], x['prior']).score, jI.judge(refI, x['text'], x['prior']).score

STREAMS = {}
def _score_on_stream(j, ref, text, prior):
    # judge()와 같은 계산을 판정기 전용 CUDA 스트림에서. 동기화는 그 스트림만 (.cpu()가 현재 스트림을 기다림)
    ctx = torch.cuda.stream(STREAMS.setdefault(id(j), torch.cuda.Stream())) if j.device == 'cuda' else contextlib.nullcontext()
    with ctx, torch.no_grad():
        a, b = j._pair(ref, text, prior)
        enc = j.tok([a], [b], truncation='only_first', max_length=j.max_len, return_tensors='pt').to(j.device)
        with torch.autocast(j.device, enabled=(j.device == 'cuda')):
            return float(j.model(**enc).logits.squeeze().float().cpu())

POOL = ThreadPoolExecutor(2)
def judge_par(jF, jI, refF, refI, x):
    fF = POOL.submit(_score_on_stream, jF, refF, x['text'], x['prior'])
    fI = POOL.submit(_score_on_stream, jI, refI, x['text'], x['prior'])
    return fF.result(), fI.result()

def bench(fn, jF, jI, inputs, refs, warm=10):
    for x in inputs[:warm]: fn(jF, jI, *refs[x['ref']], x)
    ms, sc = [], []
    for x in inputs:
        t0 = time.perf_counter()
        sf, si = fn(jF, jI, *refs[x['ref']], x)
        c = combine(sf, si)
        ms.append((time.perf_counter() - t0) * 1000); sc.append((sf, si, c))
    return np.array(ms), np.array(sc)

def bench_one(j, inputs, refs, which, warm=10):
    for x in inputs[:warm]: j.judge(refs[x['ref']][which], x['text'], x['prior'])
    return np.array([j.judge(refs[x['ref']][which], x['text'], x['prior']).judge_ms for x in inputs])
SYSTEM = '''You are an expert annotator for RAGTruth, a benchmark of hallucinations in retrieval-augmented generation.
You receive a REFERENCE that a language model was given, and the model's RESPONSE split into numbered sentences.
Label EVERY sentence against the REFERENCE only (not world knowledge).

A sentence contains a hallucination if any part of it is:
- Evident conflict: clearly contradicts the reference (wrong fact, number, name, date, negation).
- Subtle conflict: deviates from the reference in a way that changes meaning (altered term, scope, time, attribution).
- Evident baseless info: introduces information that is not in the reference, even if it may be true in the real world.
- Subtle baseless info: adds unsupported inference, opinion, or embellishment (e.g. "popular", "cozy", "family-friendly") that the reference does not support.

Not hallucinations: faithful paraphrase; reasonable summarization; statements that the reference lacks the information or declining to answer;
meta phrases such as "Based on the passages,". For Data2txt (business data + user reviews), claims supported by the structured data or by what reviewers say are supported.

Labels: "S" = no hallucination. "H" = contains a hallucination. "A" = ambiguous, careful annotators could reasonably disagree.
For every H or A sentence also give: "type" (one of: evident_conflict, subtle_conflict, evident_baseless, subtle_baseless),
"phrase" (the shortest span copied EXACTLY, character for character, from that sentence that is hallucinated), and "reason" (at most 20 words).

Return ONLY a JSON object, no other text:
{"sentences": [{"i": 1, "label": "S"}, {"i": 2, "label": "H", "type": "evident_baseless", "phrase": "...", "reason": "..."}]}
with exactly one entry per numbered sentence, in order.'''

TASK_DESC = {'QA': 'QA — answer the question using the passages',
             'Summary': 'Summary — summarize the news article',
             'Data2txt': 'Data2txt — write an overview of a business from structured data and user reviews'}

def sentences(text):
    # (시작, 끝, 문장) — 판정 지점(points_of)과 같은 pysbd 분할
    out = []
    for sp in SEG.segment(text):
        if not sp.sent.strip(): continue
        a = sp.start + (len(sp.sent) - len(sp.sent.lstrip()))
        out.append((a, sp.start + len(sp.sent.rstrip()), sp.sent.strip()))
    return out

def label_msg(c, text):
    ss = sentences(text)
    lines = '\n'.join(f'[{i + 1}] {s}' for i, (_, _, s) in enumerate(ss))
    return f"TASK: {TASK_DESC[c['task']]}\n\nREFERENCE:\n<<<\n{c['ref_label']}\n>>>\n\nRESPONSE ({len(ss)} numbered sentences):\n{lines}"

def label_params(c, text):
    # R4와 같은 요청 (모델·max_tokens·지시문)
    return {'model': LABELER, 'max_tokens': 4096, 'system': SYSTEM, 'messages': [{'role': 'user', 'content': label_msg(c, text)}]}

def parse(text, n):
    try:
        d = json.loads(text[text.find('{'): text.rfind('}') + 1])
        labs = {}
        for e in d.get('sentences', []):
            i, lab = int(e.get('i', 0)), str(e.get('label', '')).strip().upper()[:1]
            if 1 <= i <= n and lab and lab in 'SHA':
                labs[i] = {'label': lab, 'type': e.get('type', ''), 'phrase': e.get('phrase', '') or '', 'reason': e.get('reason', '')}
        return labs, len(labs) == n
    except Exception:
        return {}, False

def score_response(jF, jI, refs, text, fn):
    # 판정 지점(문장마다 8토큰째 + 문장 끝)마다 (문장 시작, 지점 끝, F1 base, I2, 조합)
    out = []
    for a, cut in points_of(text):
        t = text[a:cut].strip()
        if not t: continue
        sf, si = fn(jF, jI, refs[0], refs[1], {'text': t, 'prior': text[:a]})
        out.append((a, cut, sf, si, combine(sf, si)))
    return out

def gen_status(g):
    if g is None: return 'nogen'
    if g['stop'] == 'refusal': return 'refusal'
    if not g['text'].strip() or not sentences(g['text']): return 'empty'
    return 'valid'

MARK = '[CHECK]'

def flagged_starts(pts, thr):
    # 경보 지점(조합 점수 ≥ 임계값)이 하나라도 있는 문장의 시작 글자 위치 = 경고 문장
    return sorted({int(p[0]) for p in pts if p[4] >= thr})

def mark(text, starts):
    # 경고 문장 앞에 [CHECK] 표시 (본문 글자는 그대로)
    out, last = [], 0
    for a in sorted(starts):
        out.append(text[last:a]); out.append(MARK + ' '); last = a
    out.append(text[last:])
    return ''.join(out)

jF, jI = make_judge(F_DIR, DEVICE), make_judge(I_DIR, DEVICE)
assert jF.mode == 'retrieval' and jI.mode == 'full', (jF.mode, jI.mode)
FN = judge_par if DEVICE == 'cuda' else judge_seq

def score_texts(pairs, path):
    # pairs: key -> (근거, 글). 결과: key -> 판정 지점 목록 (드라이브에 이어서 저장)
    S = load_json(path) or {}
    t0, refs = time.time(), {}
    for k, (ref, txt) in [(k, v) for k, v in pairs.items() if k not in S]:
        if ref not in refs: refs = {ref: (jF.prepare(ref), jI.prepare(ref))}
        S[k] = [list(p) for p in score_response(jF, jI, refs[ref], txt, FN)]
        if len(S) % 25 == 0:
            save_json(S, path); print(f'  판정 {len(S)}/{len(pairs)} ({time.time() - t0:.0f}s)', flush=True)
    save_json(S, path)
    return S

SC = score_texts({it['rid']: (it['ref'], it['text']) for it in sorted(ITEMS, key=lambda x: x['sid'])}, os.path.join(AC1_DIR, 'scores_orig.json'))
FLAGS = {it['rid']: flagged_starts(SC[it['rid']], THR) for it in ITEMS if it['rid'] in SC}
nf = sum(bool(v) for v in FLAGS.values())
print(f'경고 응답 {nf}/{len(FLAGS)} ({nf / len(FLAGS):.3f}) | 과제별', {t: round(float(np.mean([bool(FLAGS[x["rid"]]) for x in ITEMS if x["task"] == t])), 3) for t in TASKS})

## 4. 고쳐 쓰기 (Haiku) — 형식 확인 → 비용 추정 → 배치

In [ ]:
import anthropic
def get_key(name):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v: return v
    except Exception:
        pass
    from getpass import getpass
    return getpass(f'{name}: ')
A = anthropic.Anthropic(api_key=get_key('ANTHROPIC_API_KEY'), max_retries=3)
PRICE = {'claude-haiku-5-5': (0.05, 0.25), 'claude-sonnet-5-5': (1.0, 5.0), 'claude-opus-5-5': (2.0, 10.0)}   # 배치 $/백만 토큰 (2026-11)

def text_of(m): return ''.join(b.text for b in m.content if getattr(b, 'type', '') == 'text')
def rec(m): return {'text': text_of(m), 'stop': m.stop_reason, 'model': m.model, 'in': m.usage.input_tokens, 'out': m.usage.output_tokens}
def cost(res): return sum(r['in'] / 1e6 * PRICE.get(r['model'], (2, 10))[0] + r['out'] / 1e6 * PRICE.get(r['model'], (2, 10))[1] for r in res.values())

def run_batch(reqs, path, chunk=400):
    # reqs: cid -> 요청. 결과: cid -> rec. 다시 실행하면 제출한 묶음·받은 결과는 건너뛴다
    res = load_json(path) or {}
    bpath = path.replace('.json', '_batches.json')
    B = load_json(bpath) or {'chunks': []}
    sent = {c for ch in B['chunks'] for c in ch['cids']}
    todo = [c for c in reqs if c not in sent and c not in res]
    for k in range(0, len(todo), chunk):
        part = todo[k:k + chunk]
        b = A.messages.batches.create(requests=[{'custom_id': c, 'params': reqs[c]} for c in part])
        B['chunks'].append({'id': b.id, 'cids': part, 'done': False}); save_json(B, bpath)
        print(f'  제출 {b.id} ({len(part)}건)', flush=True)
    for ch in B['chunks']:
        if ch.get('done'): continue
        while True:
            b = A.messages.batches.retrieve(ch['id']); rc = b.request_counts
            print(f'  {time.strftime("%H:%M:%S")} {ch["id"][-8:]} {b.processing_status} | 처리 중 {rc.processing}, 성공 {rc.succeeded}, 오류 {rc.errored}', flush=True)
            if b.processing_status == 'ended': break
            time.sleep(60)
        for r in A.messages.batches.results(ch['id']):
            if r.result.type == 'succeeded': res[r.custom_id] = rec(r.result.message)
        ch['done'] = True; save_json(B, bpath); save_json(res, path)
    miss = [c for c in reqs if c not in res]
    for c in miss[:100]:                                   # 배치 실패분은 바로 요청 1회
        try: res[c] = rec(A.messages.create(**reqs[c]))
        except Exception as e: print(f'  {c}: {type(e).__name__} {str(e)[:150]}'); break
    save_json(res, path)
    print(f'  결과 {len(res)}/{len(reqs)} | 사용량 약 ${cost({c: res[c] for c in reqs if c in res}):.2f}')
    return res


REWRITER, REWRITE_MAX, LABELER = 'claude-haiku-5-5', 2048, 'claude-opus-5-5'
_KEEP = '''Keep the original format (line breaks, lists), tone, and length as much as possible. Do not add any information that is not in the REFERENCE.
Output ONLY the revised response text, with no explanation, heading, or quotation marks.'''
TGT_SYSTEM = '''You revise a response so that every statement is supported by the REFERENCE.
In the RESPONSE, some sentences are marked with [CHECK]: an automatic checker found that they may contain information that is not supported by, or contradicts, the REFERENCE.
For each marked sentence, compare it with the REFERENCE. If it contains unsupported or contradicting content, correct that content using the REFERENCE or remove it. If the marked sentence is actually fully supported, keep it unchanged.
Do not change unmarked sentences except for minimal grammar fixes needed after your edits. Remove all [CHECK] marks.
''' + _KEEP
BLIND_SYSTEM = '''You revise a response so that every statement is supported by the REFERENCE.
Compare each sentence of the RESPONSE with the REFERENCE. If a sentence contains information that is not supported by, or contradicts, the REFERENCE, correct that content using the REFERENCE or remove it. Keep supported sentences unchanged.
''' + _KEEP

def rw_msg(it, body):
    return f"TASK: {TASK_DESC[it['task']]}\n\nREFERENCE:\n<<<\n{it['ref']}\n>>>\n\nRESPONSE:\n<<<\n{body}\n>>>"

def rw_params(system, it, body):
    return {'model': REWRITER, 'max_tokens': REWRITE_MAX, 'system': system, 'messages': [{'role': 'user', 'content': rw_msg(it, body)}]}

def rewrite_requests(items, flags):
    # tgt: 경고 응답만(표시해서) / blind: 모든 응답(표시 없이). 판정이 없는 응답은 빼고
    R = {}
    for it in items:
        st = flags.get(it['rid'])
        if st is None: continue
        if st: R[f"t-{it['rid']}"] = rw_params(TGT_SYSTEM, it, mark(it['text'], st))
        R[f"b-{it['rid']}"] = rw_params(BLIND_SYSTEM, it, it['text'])
    return R

def clean_rewrite(g):
    # 고쳐 쓴 글 (잘림·거절·빈 글은 None). 남은 [CHECK]와 감싼 <<< >>>는 지운다
    if not g or g.get('stop') in ('max_tokens', 'refusal'): return None
    t = g['text'].replace(MARK + ' ', '').replace(MARK, '').strip()
    if t.startswith('<<<'): t = t[3:]
    if t.endswith('>>>'): t = t[:-3]
    t = t.strip()
    return t if t and sentences(t) else None

import hashlib
ARMS = ['orig', 'tgt', 'blind']

LOSS_SYSTEM = '''You compare an ORIGINAL response with a REVISED version of it. The REFERENCE is the only source of truth.
Question: does the REVISED version remove or weaken any information that appears in the ORIGINAL and IS supported by the REFERENCE?
- Removing or correcting information that is NOT supported by, or contradicts, the REFERENCE is NOT a loss.
- Rewording that keeps the meaning is NOT a loss.
Return ONLY a JSON object: {"loss": "yes" | "no", "reason": "at most 20 words"}'''

def loss_params(it, orig, rev):
    u = (f"TASK: {TASK_DESC[it['task']]}\n\nREFERENCE:\n<<<\n{it['ref']}\n>>>\n\n"
         f"ORIGINAL:\n<<<\n{orig}\n>>>\n\nREVISED:\n<<<\n{rev}\n>>>")
    return {'model': LABELER, 'max_tokens': 300, 'system': LOSS_SYSTEM, 'messages': [{'role': 'user', 'content': u}]}

def parse_loss(text):
    try:
        d = json.loads(text[text.find('{'): text.rfind('}') + 1])
        v = str(d.get('loss', '')).strip().lower()
        return {'yes': True, 'no': False}.get(v)
    except Exception:
        return None

def arm_texts(items, flags, RW):
    # 응답마다 세 팔의 글. 고쳐 쓰기가 실패한 응답은 뺀다 (경고 없는 응답의 tgt = 원본)
    T = {}
    for it in items:
        st = flags.get(it['rid'])
        if st is None: continue
        b = clean_rewrite(RW.get(f"b-{it['rid']}"))
        t = clean_rewrite(RW.get(f"t-{it['rid']}")) if st else it['text']
        if b is None or t is None: continue
        T[it['rid']] = {'orig': it['text'], 'tgt': t, 'blind': b}
    return T

def hkey(kind, rid, text):
    # 같은 응답의 같은 글은 한 번만 평가 (어느 팔인지는 평가 요청에 드러나지 않음)
    return f"{kind}-{rid}-{hashlib.sha1(text.encode('utf-8')).hexdigest()[:12]}"

def eval_requests(items, T):
    F, Q, L = {}, {}, {}
    by = {it['rid']: it for it in items}
    for rid, arms in T.items():
        it = by[rid]; case = {'task': it['task'], 'ref_label': it['ref']}
        for a in ARMS:
            txt = arms[a]
            F[hkey('f', rid, txt)] = label_params(case, txt)
            if it['task'] == 'QA':
                Q[hkey('q', rid, txt)] = grade_params({'q': it['q'], 'own': it['passages']}, txt)
            if a != 'orig' and txt != arms['orig']:
                L[hkey('l', rid, txt)] = loss_params(it, arms['orig'], txt)
    return F, Q, L

def hal_of(FR, key, text):
    r = FR.get(key)
    if r is None: return None
    labs, ok = parse(r['text'], len(sentences(text)))
    return ('H' in [v['label'] for v in labs.values()]) if ok else None
GRADE_SYSTEM = '''You grade ONE answer to a question. The PASSAGES are the only source of truth.
"yes": the answer correctly answers the QUESTION with information supported by the PASSAGES (minor omissions are fine).
"partial": the answer is on the right track but misses an important part of what the PASSAGES say, or mixes correct and incorrect points.
"no": the answer does not answer the question, is wrong, or says the information is unavailable while the PASSAGES do contain it.
If the PASSAGES truly do not contain the answer, an answer that says so is "yes".
Return ONLY a JSON object: {"grade": "yes" | "partial" | "no", "reason": "at most 20 words"}'''

def grade_params(it, ans):
    u = f"QUESTION: {it['q']}\n\nPASSAGES:\n<<<\n{it['own']}\n>>>\n\nANSWER:\n<<<\n{ans}\n>>>"
    return {'model': LABELER, 'max_tokens': 300, 'system': GRADE_SYSTEM, 'messages': [{'role': 'user', 'content': u}]}

def parse_grade(text):
    try:
        d = json.loads(text[text.find('{'): text.rfind('}') + 1])
        g = str(d.get('grade', '')).strip().lower()
        return g if g in ('yes', 'partial', 'no') else None
    except Exception:
        return None


BUDGET_USD, STOP_USD = 25.0, 22.0
def est_cost(reqs, out_tok):
    tot = 0.0
    for p in reqs.values():
        n_in = (len(p.get('system', '')) + sum(len(m['content']) for m in p['messages'])) / 3.5
        o = out_tok(p) if callable(out_tok) else out_tok
        pin, pout = PRICE[p['model']]; tot += n_in / 1e6 * pin + o / 1e6 * pout
    return tot

REQ_RW = rewrite_requests(ITEMS, FLAGS)
by_rid = {it['rid']: it for it in ITEMS}
EST_RW = est_cost(REQ_RW, lambda p: len(p['messages'][0]['content']) / 3.5 * 0.4)
# 평가 예상(고쳐 쓰기 전): 충실성 = 원본 + 경고 응답 + blind 전부, 정답성 = QA × 3, 손실 = 경고 응답 + blind 전부
n_items, n_qa = len(FLAGS), sum(1 for x in ITEMS if x['task'] == 'QA')
avg_ref = np.mean([len(x['ref']) + len(x['text']) for x in ITEMS]) / 3.5
EST_EV = ((n_items * 2 + nf) * ((avg_ref + 2500) * PRICE[LABELER][0] + 400 * PRICE[LABELER][1])
          + n_qa * 3 * (avg_ref * PRICE[LABELER][0] + 80 * PRICE[LABELER][1])
          + (n_items + nf) * ((avg_ref * 1.5) * PRICE[LABELER][0] + 80 * PRICE[LABELER][1])) / 1e6
print(f'고쳐 쓰기 {len(REQ_RW)}건 예상 ${EST_RW:.2f} + 평가 약 ${EST_EV:.2f} (멈춤 기준 ${STOP_USD}, 상한 ${BUDGET_USD})')
assert EST_RW + EST_EV <= STOP_USD, '예상 비용이 멈춤 기준을 넘음 — 멈춤'

# 형식 확인: tgt·blind 1건씩 바로 (수 센트)
_k = next(k for k in REQ_RW if k.startswith('t-'))
_g = rec(A.messages.create(**REQ_RW[_k])); _it = by_rid[_k[2:]]
print('[tgt 표시된 입력]', mark(_it['text'], FLAGS[_it['rid']])[:400].replace('\n', ' '))
print('[tgt 고친 글]', (clean_rewrite(_g) or '(무효)')[:400].replace('\n', ' '), '| stop', _g['stop'])
_g = rec(A.messages.create(**REQ_RW['b-' + _it['rid']]))
print('[blind 고친 글]', (clean_rewrite(_g) or '(무효)')[:400].replace('\n', ' '), '| stop', _g['stop'])

RW = run_batch(REQ_RW, os.path.join(AC1_DIR, 'rewrite.json'))
TX = arm_texts(ITEMS, FLAGS, RW)
print(f"세 팔 확보 {len(TX)}/{len(FLAGS)} | 고친 글 무효 tgt {sum(1 for k in REQ_RW if k.startswith('t-') and clean_rewrite(RW.get(k)) is None)}, "
      f"blind {sum(1 for k in REQ_RW if k.startswith('b-') and clean_rewrite(RW.get(k)) is None)} | "
      f"원본과 같은 글 tgt {sum(1 for r, t in TX.items() if FLAGS[r] and t['tgt'] == t['orig'])}, blind {sum(1 for t in TX.values() if t['blind'] == t['orig'])}")

## 5. 평가 (Opus, 어느 팔인지 가림) — 충실성·정답성(QA)·정보 손실

In [ ]:
REQ_F, REQ_Q, REQ_L = eval_requests(ITEMS, TX)
EST2 = est_cost(REQ_F, 400) + est_cost(REQ_Q, 80) + est_cost(REQ_L, 80)
SPENT = cost(RW)
print(f'평가 요청: 충실성 {len(REQ_F)}, 정답성 {len(REQ_Q)}, 손실 {len(REQ_L)} | 예상 ${EST2:.2f} + 사용 ${SPENT:.2f}')
assert SPENT + EST2 <= BUDGET_USD - 1, '예상 비용이 상한을 넘음 — 멈춤'
FR = run_batch(REQ_F, os.path.join(AC1_DIR, 'faith.json'))
QR = run_batch(REQ_Q, os.path.join(AC1_DIR, 'grade.json'))
LR = run_batch(REQ_L, os.path.join(AC1_DIR, 'loss.json'))
# 충실성 파싱 실패는 바로 1회 재요청 (최대 50건)
text_of_key = {hkey('f', r, t[a]): t[a] for r, t in TX.items() for a in ARMS}
bad = [k for k in REQ_F if k in FR and not parse(FR[k]['text'], len(sentences(text_of_key[k])))[1]]
for k in bad[:50]:
    try: FR[k] = rec(A.messages.create(**REQ_F[k]))
    except Exception as e: print(f'  {k}: {type(e).__name__}'); break
save_json(FR, os.path.join(AC1_DIR, 'faith.json'))
print(f'충실성 파싱 실패 {len(bad)} → 재요청 후', sum(1 for k in REQ_F if k in FR and not parse(FR[k]["text"], len(sentences(text_of_key[k])))[1]))

## 6. (보고) 고친 글을 판정기로 다시 채점 — 경고가 사라졌나

In [ ]:
RS = score_texts({r: (by_rid[r]['ref'], t['tgt']) for r, t in TX.items() if FLAGS[r] and t['tgt'] != t['orig']},
                 os.path.join(AC1_DIR, 'scores_tgt.json'))
RESCORE = {r: bool(flagged_starts(p, THR)) for r, p in RS.items()}
print(f'고친 tgt {len(RESCORE)}개 중 다시 경고 {sum(RESCORE.values())}')

## 7. 사전 기준 판정

In [ ]:
def build_rows(items, flags, T, FR, QR, LR, rescore=None):
    rows = []
    for it in items:
        rid = it['rid']; st = flags.get(rid)
        row = {'rid': rid, 'task': it['task'], 'gen_model': it['gen_model'], 'scored': st is not None,
               'flagged': bool(st), 'n_flag_sent': len(st or []), 'n_sent': len(sentences(it['text'])), 'in_T': rid in T}
        if rid in T:
            arms = T[rid]
            for a in ARMS:
                txt = arms[a]
                row[f'hal_{a}'] = hal_of(FR, hkey('f', rid, txt), txt)
                row[f'len_{a}'] = len(txt)
                row[f'changed_{a}'] = txt != arms['orig']
                if it['task'] == 'QA':
                    q = QR.get(hkey('q', rid, txt)); row[f'grade_{a}'] = parse_grade(q['text']) if q else None
                if a != 'orig':
                    if txt == arms['orig']:
                        row[f'loss_{a}'] = False                     # 바뀌지 않으면 손실 없음
                    else:
                        l = LR.get(hkey('l', rid, txt)); row[f'loss_{a}'] = parse_loss(l['text']) if l else None
            if rescore is not None and rid in rescore:
                row['reflag_tgt'] = rescore[rid]
        rows.append(row)
    D = pd.DataFrame(rows)
    need = [f'hal_{a}' for a in ARMS] + ['loss_tgt', 'loss_blind']
    ok = D['in_T'].astype(bool) & D.reindex(columns=need).notna().all(axis=1)
    qa = (D.task == 'QA').values
    gq = D.reindex(columns=[f'grade_{a}' for a in ARMS]).notna().all(axis=1).values
    D['valid'] = ok.values & (~qa | gq)
    return D

def boot(x, n=2000, seed=SEED_AC):
    # 응답 단위 쌍체 부트스트랩 95% 구간
    x = np.asarray(x, float)
    if len(x) == 0: return (float('nan'), float('nan'))
    rng = np.random.default_rng(seed)
    vals = [x[rng.integers(0, len(x), len(x))].mean() for _ in range(n)]
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def judge_ac1(D):
    d = D[D.valid].copy(); n = len(d)
    h = {a: d[f'hal_{a}'].astype(float) for a in ARMS}
    p = {a: float(h[a].mean()) if n else float('nan') for a in ARMS}
    out = {'n': n, 'n_items': len(D), 'n_flagged_valid': int(d.flagged.sum()), 'n_flagged_all': int(D.flagged.sum()), 'hal': p}
    for a in ['tgt', 'blind']:
        x = h[a] - h['orig']
        out[f'diff_{a}'], out[f'diff_{a}_ci'] = float(x.mean()), boot(x)
        out[f'rel_{a}'] = out[f'diff_{a}'] / p['orig'] if p['orig'] > 0 else float('nan')
    q = d[d.task == 'QA']
    out['qa_n'] = len(q)
    yes = {a: (q[f'grade_{a}'] == 'yes').astype(float) for a in ARMS}
    out['yes'] = {a: float(yes[a].mean()) if len(q) else float('nan') for a in ARMS}
    for a in ['tgt', 'blind']:
        x = yes[a] - yes['orig']
        out[f'yes_diff_{a}'], out[f'yes_diff_{a}_ci'] = float(x.mean()) if len(q) else float('nan'), boot(x)
    for a in ['tgt', 'blind']:
        ch = d[d[f'changed_{a}'].astype(bool)]
        out[f'n_changed_{a}'] = len(ch)
        out[f'loss_{a}'] = float(ch[f'loss_{a}'].astype(float).mean()) if len(ch) else 0.0
    out['rewrite_frac'] = float(d.flagged.mean()) if n else float('nan')
    out['gain_tgt'], out['gain_blind'] = p['orig'] - p['tgt'], p['orig'] - p['blind']
    out['AC1_0'] = bool(n >= 405 and out['n_flagged_valid'] >= 60)
    out['AC1_1'] = bool(out['rel_tgt'] <= -0.15 and out['diff_tgt_ci'][1] < 0)
    out['AC1_2'] = bool(out['yes_diff_tgt'] >= -0.05 and out['yes_diff_tgt_ci'][0] >= -0.10 and out['loss_tgt'] <= 0.15)
    out['AC1_3'] = bool(out['rewrite_frac'] <= 0.30 and out['gain_tgt'] >= 0.5 * out['gain_blind'])
    out['AC1'] = out['AC1_0'] and out['AC1_1'] and out['AC1_2'] and out['AC1_3']
    # 보고만
    out['by_task'] = {t: {a: float(g[f'hal_{a}'].astype(float).mean()) for a in ARMS} | {'n': len(g), 'flag_frac': float(g.flagged.mean())}
                      for t, g in d.groupby('task')}
    out['hal_orig_flagged'] = float(d[d.flagged].hal_orig.astype(float).mean()) if out['n_flagged_valid'] else float('nan')
    out['hal_orig_unflagged'] = float(d[~d.flagged].hal_orig.astype(float).mean()) if n - out['n_flagged_valid'] else float('nan')
    out['len_ratio'] = {a: float((d[f'len_{a}'] / d.len_orig).median()) for a in ['tgt', 'blind']}
    if 'reflag_tgt' in d:
        r = d[d.flagged & d.changed_tgt.astype(bool)].reflag_tgt.dropna()
        out['reflag_tgt'] = float(r.astype(float).mean()) if len(r) else float('nan')
    return out

D = build_rows(ITEMS, FLAGS, TX, FR, QR, LR, RESCORE)
R = judge_ac1(D)
f = lambda ci: f'[{ci[0]:+.3f}, {ci[1]:+.3f}]'
print(f"유효 {R['n']}/{R['n_items']} | 경고 응답 {R['n_flagged_valid']} (전체 {R['n_flagged_all']}) | 고친 응답 비율 {R['rewrite_frac']:.3f}")
print(f"환각 응답 비율: orig {R['hal']['orig']:.3f} | tgt {R['hal']['tgt']:.3f} | blind {R['hal']['blind']:.3f}")
print(f"  tgt − orig {R['diff_tgt']:+.3f} {f(R['diff_tgt_ci'])} (상대 {R['rel_tgt']:+.1%}) | blind − orig {R['diff_blind']:+.3f} {f(R['diff_blind_ci'])} (상대 {R['rel_blind']:+.1%})")
print(f"QA yes ({R['qa_n']}개): orig {R['yes']['orig']:.3f} | tgt {R['yes']['tgt']:.3f} | blind {R['yes']['blind']:.3f} | tgt − orig {R['yes_diff_tgt']:+.3f} {f(R['yes_diff_tgt_ci'])}")
print(f"정보 손실: tgt {R['loss_tgt']:.3f} ({R['n_changed_tgt']}개 고침) | blind {R['loss_blind']:.3f} ({R['n_changed_blind']}개)")
print(f"\n=== AC1-0 유효성(유효 ≥ 405, 경고 응답 ≥ 60) → {'충족' if R['AC1_0'] else '미달(판정 보류)'} ===")
print(f"=== AC1-1 효과: 상대 {R['rel_tgt']:+.1%} (≤ −15%), 상한 {R['diff_tgt_ci'][1]:+.3f} (< 0) → {'충족' if R['AC1_1'] else '미달'} ===")
print(f"=== AC1-2 부작용: yes {R['yes_diff_tgt']:+.3f} (≥ −0.05), 하한 {R['yes_diff_tgt_ci'][0]:+.3f} (≥ −0.10), 손실 {R['loss_tgt']:.3f} (≤ 0.15) → {'충족' if R['AC1_2'] else '미달'} ===")
print(f"=== AC1-3 판정기의 가치: 고친 비율 {R['rewrite_frac']:.3f} (≤ 0.30), 감소폭 tgt {R['gain_tgt']:.3f} vs blind의 절반 {0.5 * R['gain_blind']:.3f} → {'충족' if R['AC1_3'] else '미달'} ===")
print(f"=== AC1 {'채택' if R['AC1'] else '미채택'} ===")
print('(보고) 과제별', json.dumps(R['by_task'], ensure_ascii=False))
print(f"(보고) 원본 환각 비율: 경고 응답 {R['hal_orig_flagged']:.3f} / 경고 없는 응답 {R['hal_orig_unflagged']:.3f} | 길이 비율 {R['len_ratio']} | 고친 tgt 다시 경고 {R.get('reflag_tgt')}")

## 8. 저장

In [ ]:
usage = {f: round(cost(load_json(os.path.join(AC1_DIR, f))), 2) for f in ['rewrite.json', 'faith.json', 'grade.json', 'loss.json']
         if os.path.exists(os.path.join(AC1_DIR, f))}
print('API 사용량(배치 단가) $', usage, '합계', round(sum(usage.values()), 2))
LOCAL = '/content/ac1_local'; os.makedirs(LOCAL, exist_ok=True)
D.to_csv(os.path.join(LOCAL, 'ac1_rows.csv'), index=False)            # 원문 없음
json.dump(dict(R, cost_usd=usage, gpu=GPU, threshold=THR, rewriter=REWRITER, labeler=LABELER, seed=SEED_AC),
          open(os.path.join(LOCAL, 'ac1_summary.json'), 'w'), ensure_ascii=False, indent=1, default=float)
for f in os.listdir(LOCAL): shutil.copy(os.path.join(LOCAL, f), os.path.join(AC1_DIR, f))
zp = shutil.make_archive('/content/ac1_results', 'zip', LOCAL)
from google.colab import files
files.download(zp)